<a href="https://colab.research.google.com/github/Simran-12005/TableTalk_GSoc_2026/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Audio_Speech_Actors_01-24.zip to Audio_Speech_Actors_01-24.zip


In [4]:
import os

os.makedirs("dataset", exist_ok=True)

for file in uploaded.keys():
    os.rename(file, f"dataset/{file}")

In [5]:
import librosa
import numpy as np
import pandas as pd
import os

DATASET_PATH = "/content/dataset"

data = []

# Extract emotion label from filename (RAVDESS)
def get_label(filename):
    try:
        return int(filename.split("-")[2])
    except:
        return -1  # fallback if format differs

def extract_features(file_path):
    # Load audio (fixed duration)
    y, sr = librosa.load(file_path, duration=3)

    # Normalize
    y = librosa.util.normalize(y)

    # MFCC (13)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)

    # Pitch
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0

    # Energy
    energy = np.mean(librosa.feature.rms(y=y))

    # Spectral centroid
    spec_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))

    # Duration
    duration = librosa.get_duration(y=y, sr=sr)

    return list(mfcc) + [pitch, energy, spec_centroid, duration]


# Process all files
for file in os.listdir(DATASET_PATH):
    if file.endswith(".wav"):
        file_path = os.path.join(DATASET_PATH, file)

        features = extract_features(file_path)
        label = get_label(file)

        data.append([file, label] + features)

# Columns
columns = ["filename", "label"] + [f"mfcc_{i}" for i in range(13)] + [
    "pitch", "energy", "spectral_centroid", "duration"
]

# DataFrame
df = pd.DataFrame(data, columns=columns)

# Save CSV
df.to_csv("features.csv", index=False)

print("✅ Task 1 Completed Successfully!")
df.head()

✅ Task 1 Completed Successfully!


,filename,label,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,pitch,energy,spectral_centroid,duration
